In [3]:
import os
import glob
import shutil
import pandas as pd
from datetime import datetime, timedelta

# ---------------------------------------------
# CONFIGURATION
# ---------------------------------------------
VISIT_GAP_THRESHOLD_SECONDS = 3
GAP_THRESHOLD = timedelta(seconds=VISIT_GAP_THRESHOLD_SECONDS)
MINIMUM_VISIT_DURATION = timedelta(seconds=60)

# *** NEW DATE WINDOW ***
DATE_START = datetime(2025, 11, 2, 0, 0, 0)
DATE_END   = datetime(2025, 11, 2, 23, 59, 59)

BUFFER = timedelta(minutes=0.1)
OUTPUT_ROOT = r"D:\Day2_Visit"   # <--- NEW structured per-visit output

# CAMERA → FEEDER MAPPING
CAMERA_MAPPING = {
    "Camera1": "2C",
    "Camera2": "2A",
    "Camera3": "2B"
}

CAMERA_DIRS = {
    "Camera1": r"D:\Camera1_transfer_11_04_2025",
    "Camera2": r"D:\Camera2_transfer_11_04_2025",
    "Camera3": r"D:\Camera3_transfer_11_04_2025"
}

ACTUAL_TIME = "2025-10-31 11:47:00"
CAMERA_TIME = "1969-12-31 18:04:00"

CSV_FILE = r"C:\Users\spaudel\Downloads\Sow Export 10_31_11_4.xlsx"

# =========================================================
# 1️⃣ LOAD & FILTER FEEDER CSV
# =========================================================
def load_and_filter_feeder_data(filepath, feeder_id):
    df = pd.read_excel(filepath) if filepath.lower().endswith(("xls", "xlsx")) else pd.read_csv(filepath)

    df.columns = df.columns.str.strip()
    df["Start"] = pd.to_datetime(df["Start"])
    df["End"]   = pd.to_datetime(df["End"])
    df["FEEDER_STR"] = df["FEEDER"].astype(str).str.strip()

    return df[df["FEEDER_STR"] == str(feeder_id).strip()]

# =========================================================
# 2️⃣ CAMERA VISIT EXTRACTION
# =========================================================
def get_real_capture_details(filename, time_offset, root):
    parts = filename.split("_")
    if len(parts) < 5:
        return None

    date_str = parts[1]
    time_str = parts[2].replace("-", ":")
    frame_num = parts[3]

    timestamp = f"{date_str} {time_str}"

    try:
        recorded = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        real_time = recorded + time_offset
        return {
            "filename": filename,
            "real_time": real_time,
            "frame_number": int(frame_num),
            "filepath": os.path.join(root, filename),
        }
    except:
        return None


def extract_camera_visits(directory, actual_time, camera_time):
    actual = datetime.strptime(actual_time, "%Y-%m-%d %H:%M:%S")
    cam = datetime.strptime(camera_time, "%Y-%m-%d %H:%M:%S")
    time_offset = actual - cam

    files = glob.glob(os.path.join(directory, "*_depth.png"))
    parsed = []

    for f in files:
        d = get_real_capture_details(os.path.basename(f), time_offset, directory)
        if d:
            parsed.append(d)

    parsed.sort(key=lambda x: x["real_time"])

    raw_visits = []
    current = []

    for i, f in enumerate(parsed):
        if i == 0:
            current.append(f)
            continue

        gap = f["real_time"] - parsed[i - 1]["real_time"]
        if gap > GAP_THRESHOLD:
            raw_visits.append(current)
            current = [f]
        else:
            current.append(f)

    if current:
        raw_visits.append(current)

    # Filter short visits
    long_visits = [v for v in raw_visits if (v[-1]["real_time"] - v[0]["real_time"]) >= MINIMUM_VISIT_DURATION]

    # *** NEW FILTER: only keep visits fully inside the DATE WINDOW ***
    final = []
    for v in long_visits:
        start_t = v[0]["real_time"]
        if DATE_START <= start_t <= DATE_END:
            final.append(v)

    return final, parsed

# =========================================================
# 3️⃣ MATCH CAMERA VISITS ↔ SOW IDs USING BUFFER
# =========================================================
def match_visits_to_sows(visits, feeder_df):
    results = []

    for i, v in enumerate(visits, start=1):
        cam_start = v[0]["real_time"]
        cam_end   = v[-1]["real_time"]

        matches = feeder_df[
            (feeder_df["Start"] <= cam_end + BUFFER) &
            (feeder_df["End"]   >= cam_start - BUFFER)
        ]

        sow_ids = matches["SOW"].unique().tolist() if len(matches) else []

        results.append({
            "VisitID": i,
            "CameraStart": cam_start,
            "CameraEnd": cam_end,
            "Frames": len(v),
            "SOW_IDs": sow_ids
        })

    return results

# =========================================================
# 4️⃣ COPY VISITS (ONE SOW ID ONLY)
# =========================================================
def copy_single_id_visits(cam_name, visits, matches, writer_rows):
    for v, m in zip(visits, matches):

        sow_ids = m["SOW_IDs"]
        if len(sow_ids) != 1:
            continue

        sow_id = str(sow_ids[0])
        start_t = m["CameraStart"].strftime("%Y-%m-%d_%H-%M-%S")
        end_t   = m["CameraEnd"].strftime("%Y-%m-%d_%H-%M-%S")

        # Create visit folder
        visit_folder = os.path.join(
            OUTPUT_ROOT,
            f"{cam_name}_{start_t}__{end_t}"
        )
        os.makedirs(visit_folder, exist_ok=True)

        # Copy all frames in the visit
        for frame in v:
            shutil.copy2(frame["filepath"], os.path.join(visit_folder, frame["filename"]))

        # Record CSV row
        writer_rows.append([
            cam_name,
            m["CameraStart"],
            m["CameraEnd"],
            sow_id,
            len(v),
            visit_folder
        ])

# =========================================================
# 5️⃣ MASTER LOOP
# =========================================================
if __name__ == "__main__":

    csv_rows = []

    for cam_name, feeder_id in CAMERA_MAPPING.items():
        print("\n============================================")
        print(f"PROCESSING {cam_name}  (Feeder {feeder_id})")
        print("============================================")

        feeder_df = load_and_filter_feeder_data(CSV_FILE, feeder_id)
        visits, all_frames = extract_camera_visits(CAMERA_DIRS[cam_name], ACTUAL_TIME, CAMERA_TIME)
        matches = match_visits_to_sows(visits, feeder_df)

        # Copy visits with exactly ONE Sow ID
        copy_single_id_visits(cam_name, visits, matches, csv_rows)

    # -------------------------------
    # Save summary CSV
    # -------------------------------
    out_csv = os.path.join(OUTPUT_ROOT, "matched_visits_summary.csv")
    df_out = pd.DataFrame(csv_rows, columns=[
        "Camera", "StartTime", "EndTime", "SowID", "NumFrames", "Folder"
    ])
    df_out.to_csv(out_csv, index=False)

    print("\nDONE! CSV saved at:", out_csv)



PROCESSING Camera1  (Feeder 2C)

PROCESSING Camera2  (Feeder 2A)

PROCESSING Camera3  (Feeder 2B)

DONE! CSV saved at: D:\Day2_Visit\matched_visits_summary.csv
